Read the 2.41-million-record consolidated screening source once and create reusable, validated handoffs for downstream analysis

Two grains are built without aggregation or explosion: 
- a durable compact one-row-per-screened-`UT` evidence table for screening analyses, and 
- a one-row-per-eligible-`UT` bibliographic/screening build cache that seeds the integrated biodiversity-evidence corpus. 

In [1]:
import sys
from pathlib import Path

import pandas as pd

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "data_helpers").is_dir() and (candidate / "checklists").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the biodiversity repository root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_helpers import results_config
from data_helpers.prep import evidence_corpus_prep
from data_helpers._config import get_merged_corpus_config


In [2]:
RESULTS_CONFIG = results_config.load_results_config()
CONFIG = RESULTS_CONFIG["screening_analysis_prep"]
CORPUS_CONFIG = get_merged_corpus_config()
STORE = evidence_corpus_prep.ScreeningPreparedStore(PROJECT_ROOT / CONFIG["output_directory"])


The source is streamed in configured chunks. `UT` must be unique and non-null across all screened records; the eligible artifact must equal the eligible key subset in the same order.

In [3]:
bundle = evidence_corpus_prep.build_screening_preparation(
    PROJECT_ROOT / CONFIG["screening_source"],
    eligible_columns=CORPUS_CONFIG["screening"]["columns"],
    chunksize=CONFIG["chunksize"],
    repository_root=PROJECT_ROOT,
)
summary = pd.Series(bundle.manifest["summary"], name="value").rename_axis("metric").reset_index()
display(summary.loc[~summary["metric"].isin(["raw_columns"])].style.format({"value": lambda value: f"{value:,}" if isinstance(value, int) else value}).hide(axis="index"))
display(pd.DataFrame(bundle.manifest["eligible_direction_audit"]).style.format({"records": "{:,}", "percent_of_eligible": "{:.2f}%"}).hide(axis="index"))
display(bundle.exclusion_overlap[["intersection", "combination", "records", "percent"]].style.format({"records": "{:,}", "percent": "{:.2f}%"}).hide(axis="index"))


metric,value
screened_records,"2,414,191"
raw_column_count,24
eligible_records,"605,199"
not_eligible_records,"1,808,992"
eligibility_label_eligible,"605,199"
eligibility_label_not_eligible,"1,808,992"
eligible_with_unclear_step,"121,707"
eligible_negative,"253,081"
eligible_not_negative,"352,118"
complete_ineligible_with_zero_reason,"1,808,992"


direction_of_change,records,percent_of_eligible
negative,"253,081",41.82%
mixed,"141,267",23.34%
positive,"112,684",18.62%
unclear,"98,167",16.22%


intersection,combination,records,percent
1,Q1: biodiversity change + Q2: direction of change + Q3: anthropogenic driver + Q4: driver-change linkage,"1,081,249",59.77%
2,Q1: biodiversity change + Q2: direction of change + Q4: driver-change linkage,"435,966",24.10%
3,Q3: anthropogenic driver + Q4: driver-change linkage,"217,696",12.03%
4,Q1: biodiversity change + Q2: direction of change,"35,857",1.98%
5,Q4: driver-change linkage,"13,742",0.76%
6,Q3: anthropogenic driver,"11,332",0.63%
7,Q1: biodiversity change,"6,645",0.37%
8,Q1: biodiversity change + Q3: anthropogenic driver + Q4: driver-change linkage,"2,913",0.16%
9,Q1: biodiversity change + Q4: driver-change linkage,"1,411",0.08%
10,Q2: direction of change,948,0.05%


The manifest is the compact screening-results handoff. The results notebook loads only the manifest and overlap CSV. 

In [4]:
written = STORE.write(bundle)
reloaded = STORE.load()
assert reloaded.screening["UT"].equals(bundle.screening["UT"])
assert reloaded.eligible["UT"].equals(bundle.eligible["UT"])
pd.testing.assert_frame_equal(reloaded.exclusion_overlap, bundle.exclusion_overlap, check_dtype=False)
print("Verified screening bundle:")
for path in written:
    print(f"  {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size / 1024**2:.2f} MiB)")


Verified screening bundle:
  notebooks/data_processing/outputs/03_screening_analysis_prep/screening_publications.parquet (9.30 MiB)
  notebooks/data_processing/outputs/03_screening_analysis_prep/eligible_screening_publications.parquet (405.45 MiB)
  notebooks/data_processing/outputs/03_screening_analysis_prep/screening_exclusion_overlap.csv (0.00 MiB)
  notebooks/data_processing/outputs/03_screening_analysis_prep/manifest.json (0.00 MiB)
